# Custom Wake Word Trainer (openWakeWord)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/contextuai/hey-spark/blob/main/tools/wakeword_trainer.ipynb)

Part of [Hey Spark, by ContextuAI](https://github.com/contextuai/hey-spark). Step-by-step guide: [`docs/wakeword.md`](https://github.com/contextuai/hey-spark/blob/main/docs/wakeword.md).

Trains a custom wake word model and gives you `.onnx` and `.tflite` files.

**It's free.** Everything runs on Colab's free T4 GPU: no Colab Pro, no payment, no API keys,
and no recordings of your voice. All you need is a Google account.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU → Save.** (Not TPU; the training code can't use it.)
2. Change the wake word in **Step 6 · Settings**. You don't need to edit anything else.
3. Run the cells **in order, top to bottom**: click ▶ on each one, or use **Runtime → Run all**
   and listen to the test clip when Step 8 plays it.

**Free-tier tips**
- Free GPU time is limited per day. If Colab says it **can't connect to a GPU backend**, try
  again later; one training run fits comfortably in a day's allowance.
- Keep this tab open and check on it now and then: Colab disconnects sessions that sit idle.
- Colab deletes the session's files when it disconnects. Download your model (Step 11) as soon
  as training finishes, or tick `save_to_drive` in Step 13.
- The run needs about 25 GB of disk, which the free runtime has. Step 1 checks it.

**Time:** about 10 minutes of setup (Steps 2–4), 20–40 minutes to download training data (Step 9, first time only), then 30–60 minutes of training on a T4.

*The background audio this downloads has mixed licenses, so models you train are for **non-commercial personal use only**.*

*Built on [openWakeWord](https://github.com/dscripka/openWakeWord) (Apache-2.0) and [piper-sample-generator](https://github.com/rhasspy/piper-sample-generator) (MIT).*

In [ ]:
# @title 1 · Check the GPU and disk
import shutil
import subprocess
r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    print("✅ GPU:", r.stdout.strip())
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")
    print("   (It still works on CPU, but slowly. If you stay on CPU, lower the numbers in Step 6.)")

free_gb = shutil.disk_usage(".").free / 1e9
if free_gb >= 25:
    print(f"✅ Disk: {free_gb:.0f} GB free")
else:
    print(f"⚠️  Only {free_gb:.0f} GB free; training needs about 25 GB.")
    print("   Runtime → Disconnect and delete runtime, then start again for a fresh disk.")


In [ ]:
# @title 2 · Install packages  (~5 min, first time only)
# @markdown Clones openWakeWord and piper-sample-generator and installs every Python package
# @markdown the training script needs. Safe to re-run. If Colab offers to **restart the session**,
# @markdown you can ignore it: training runs in its own process and sees the new packages.
import subprocess
import sys
from pathlib import Path


def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


PIP = f"{sys.executable} -m pip install -q"

if not Path("openwakeword").exists():
    sh("git clone -q https://github.com/dscripka/openwakeword")
if not Path("piper-sample-generator").exists():
    sh("git clone -q https://github.com/rhasspy/piper-sample-generator")
    sh("cd piper-sample-generator && git -c advice.detachedHead=false checkout -q 213d4d5")

sh(f"{PIP} -e ./openwakeword --no-deps")
sh(f"{PIP} mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 "
   "audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 "
   "pronouncing==0.2.0 deep-phonemizer==0.0.19 soundfile")
sh(f"{PIP} onnx2tf onnx_graphsurgeon sng4onnx")  # pins its own onnx/onnxruntime/litert/onnxsim
sh(f'{PIP} "datasets>=3,<4"')                    # 4.x decodes audio via torchcodec; breaks row['audio']['array']
sh(f"{PIP} piper-tts webrtcvad")                 # abi3 wheel works on Py3.13; bundles espeak-ng
# No onnx-tf: it imports onnx.mapping, which current onnx removed. Step 3 disables train.py's
# onnx-tf export; oww_train.py makes the .tflite with onnx2tf instead.
print("\n✅ Packages installed")

In [ ]:
# @title 3 · Apply compatibility patches
# @markdown Small fixes so the older training code runs on Colab's Python 3.13, SciPy,
# @markdown torch ≥ 2.6 and torchaudio ≥ 2.9. Safe to re-run.
import importlib
import importlib.util
import re
from pathlib import Path

TORCHAUDIO_SHIM = '''
import sys
from dataclasses import dataclass
import soundfile as _sf
import torch as _torch
_ta = sys.modules["torchaudio"]

@dataclass
class AudioMetaData:
    sample_rate: int
    num_frames: int
    num_channels: int
    bits_per_sample: int = 16
    encoding: str = "PCM_S"

def info(uri, *a, **k):
    i = _sf.info(str(uri))
    return AudioMetaData(i.samplerate, i.frames, i.channels)

def load(uri, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, *a, **k):
    data, sr = _sf.read(uri if hasattr(uri, "read") else str(uri),
                        start=frame_offset, frames=num_frames, dtype="float32", always_2d=True)
    t = _torch.from_numpy(data)
    return (t.T.contiguous() if channels_first else t), sr

def save(uri, src, sample_rate, channels_first=True, *a, **k):
    x = src.detach().cpu().numpy()
    _sf.write(str(uri), x.T if channels_first else x, sample_rate)

if not hasattr(_ta, "set_audio_backend"):
    _ta.set_audio_backend = lambda backend=None: None
if not hasattr(_ta, "info"):
    _ta.info, _ta.AudioMetaData = info, AudioMetaData
if tuple(int(v) for v in _ta.__version__.split("+")[0].split(".")[:2]) >= (2, 9):
    _ta.load, _ta.save = load, save
'''

PIPER_PHONEMIZE_SHIM = '''\
from piper.phonemize_espeak import EspeakPhonemizer
_p = None
def phonemize_espeak(text, voice):
    global _p
    if _p is None:
        _p = EspeakPhonemizer()
    return _p.phonemize(voice, text)
'''


def pkg_dir(name):
    importlib.invalidate_caches()
    spec = importlib.util.find_spec(name)
    assert spec is not None, f"Package '{name}' isn't installed. Run Step 2 first."
    return Path(spec.submodule_search_locations[0])


# acoustics.directivity uses removed scipy APIs; oWW only needs acoustics.generator
init = pkg_dir("acoustics") / "__init__.py"
s = init.read_text()
if "import acoustics.directivity\n" in s and "# oww" not in s:
    init.write_text(s.replace("import acoustics.directivity\n",
                              "try:\n    import acoustics.directivity\nexcept ImportError: pass  # oww\n"))
print("✓ acoustics")

# torchaudio>=2.9: restore set_audio_backend / info, route load/save through soundfile
ta = pkg_dir("torchaudio")
(ta / "_oww_compat.py").write_text(TORCHAUDIO_SHIM)
ta_init = ta / "__init__.py"
if "_oww_compat" not in ta_init.read_text():
    with open(ta_init, "a") as f:
        f.write("\nfrom . import _oww_compat  # openWakeWord notebook shim\n")
print("✓ torchaudio")

# piper-sample-generator: phonemize via piper-tts, and torch>=2.6 defaults weights_only=True
piper = Path("piper-sample-generator")
assert piper.exists(), "piper-sample-generator is missing. Run Step 2 first."
(piper / "piper_phonemize.py").write_text(PIPER_PHONEMIZE_SHIM)
gs = piper / "generate_samples.py"
g = gs.read_text()
if "weights_only=False" not in g:
    gs.write_text(g.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)"))
print("✓ piper-sample-generator")

# train.py's built-in TFLite export needs onnx-tf, which breaks on current onnx (no onnx.mapping)
# and makes --train_model exit 1 after saving the .onnx. oww_train.py converts with onnx2tf
# instead, so turn the upstream step into a no-op.
tp = Path("openwakeword/openwakeword/train.py")
assert tp.exists(), "openwakeword is missing. Run Step 2 first."
t = tp.read_text()
if "# oww: skip onnx_tf" not in t:
    t, n = re.subn(r"^(\s*)from onnx_tf\.backend import prepare",
                   r"\1return  # oww: skip onnx_tf\n\1from onnx_tf.backend import prepare",
                   t, count=1, flags=re.M)
    assert n == 1, "train.py layout differs; grep for onnx_tf and patch by hand."
    tp.write_text(t)
print("✓ train.py (TFLite via onnx2tf)")
print("\n✅ Patches applied")

In [ ]:
# @title 4 · Download the feature and voice models  (~1 min)
# @markdown openWakeWord's melspectrogram and embedding models, plus the Piper voice used to
# @markdown synthesize training clips. Files that already exist are skipped.
import subprocess
from pathlib import Path

OWW_MODELS = Path("openwakeword/openwakeword/resources/models")
OWW_MODELS.mkdir(parents=True, exist_ok=True)
FILES = {
    OWW_MODELS / f: f"https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{f}"
    for f in ["embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"]
}
FILES[Path("piper-sample-generator/models/en_US-libritts_r-medium.pt")] = (
    "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt")

for dest, url in FILES.items():
    if dest.exists() and dest.stat().st_size > 0:
        print("✓ already have", dest.name)
        continue
    print("downloading", dest.name, flush=True)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    subprocess.run(["wget", "-q", "-O", str(part), url], check=True)
    part.rename(dest)
print("\n✅ Models ready")

## 5 · Save the training script
Just run this cell. It saves the script as `oww_train.py` and should print **`Writing oww_train.py`** (or `Overwriting`). You don't need to read or change it.

In [ ]:
%%writefile oww_train.py
#!/usr/bin/env python3
"""
oww_train.py - end-to-end openWakeWord custom model training for Google Colab (Python 3.13).

Run from a Colab cell (GPU runtime strongly recommended):

    !python oww_train.py --word "hey spark" --test-only    # synthesize test_generation.wav and stop
    !python oww_train.py --data-only --fma-hours 3       # download background/RIR/feature data and stop
    !python oww_train.py --word "hey spark"              # full pipeline
    !python oww_train.py --word "Kyla" --pron "kyla=K AY1 L AH0"   # word not in CMUdict

Then, back in the notebook:

    from IPython.display import Audio; Audio("test_generation.wav")   # preview the TTS
    from google.colab import files
    files.download("my_custom_model/hey_spark.onnx")
    files.download("my_custom_model/hey_spark.tflite")

Packages, compat patches and the feature/TTS models are set up by the notebook cells before
this one (Steps 2-4); this script checks for them and stops if they are missing.
Every stage is idempotent: re-running skips work that is already done.
Because it runs as a subprocess, no "Runtime -> Restart session" is needed after installs.

Note: the downloaded background/RIR data has mixed licenses; models trained with it are
for non-commercial personal use only (same as the upstream example notebook).
"""

import argparse
import re
import shutil
import subprocess
import sys
from pathlib import Path

# ----------------------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------------------
ROOT = Path.cwd()
OWW = ROOT / "openwakeword"
OWW_PKG = OWW / "openwakeword"
OWW_MODELS = OWW_PKG / "resources" / "models"
PIPER = ROOT / "piper-sample-generator"
PIPER_MODEL = PIPER / "models" / "en_US-libritts_r-medium.pt"
OUT = ROOT / "my_custom_model"

# Folders handed to openWakeWord's train.py. It loads EVERY file inside them (no extension
# filter), so nothing but .wav files may live in these folders.
RIR_DIR = ROOT / "mit_rirs"
AUDIOSET_DIR = ROOT / "audioset_16k"
FMA_DIR = ROOT / "fma"
AUDIO_DIRS = [RIR_DIR, AUDIOSET_DIR, FMA_DIR]

# Pronunciations for words missing from CMUdict (lowercase word -> list of ARPAbet strings).
# DeepPhonemizer's S3 checkpoint now returns 403, so any out-of-vocabulary word MUST be here.
# Add more with --pron word=ARPABET, or here, e.g. "kyla": ["K AY1 L AH0"].
CUSTOM_WORDS: dict[str, list[str]] = {}

# ----------------------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------------------
def log(msg):
    print(f"\n{'=' * 72}\n{msg}\n{'=' * 72}", flush=True)


def run(cmd, check=True):
    """Run a shell command, streaming output. Abort the script if it fails (when check=True)."""
    print(f"$ {cmd}", flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    if rc != 0 and check:
        sys.exit(f"\n❌ Command failed (exit {rc}): {cmd}\n   Scroll up for its traceback.")
    return rc == 0


def require(*paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        sys.exit("\n❌ Expected output missing:\n   " + "\n   ".join(missing))


def done_marker(d):
    """Completion marker kept OUTSIDE the data folder (train.py loads every file inside it).
    e.g. ./mit_rirs -> ./.mit_rirs.complete. Migrates markers left inside by older runs."""
    d = Path(d)
    m = d.parent / f".{d.name}.complete"
    old = d / ".complete"
    if old.exists():
        old.unlink()
        m.touch()
        print(f"✓ Moved completion marker out of {d.name}/ -> {m.name}")
    return m


def sanitize_audio_dirs(dirs):
    """Remove anything that isn't a .wav file, so train.py never tries to load it as audio."""
    for d in dirs:
        d = Path(d)
        if not d.exists():
            continue
        for p in d.iterdir():
            if p.is_file() and p.suffix.lower() != ".wav":
                print(f"  removing non-audio file: {p}")
                p.unlink()


# ----------------------------------------------------------------------------------------
# Stage 1: confirm the notebook's install / patch / model cells have run
# ----------------------------------------------------------------------------------------
def check_setup():
    missing = [str(p) for p in (
        OWW_PKG / "train.py",
        *(OWW_MODELS / f for f in ("embedding_model.onnx", "melspectrogram.onnx")),
        PIPER / "generate_samples.py",
        PIPER / "piper_phonemize.py",
        PIPER_MODEL,
    ) if not p.exists()]
    if missing:
        sys.exit("\n❌ Setup incomplete. Run notebook Steps 2-4 first. Missing:\n   " + "\n   ".join(missing))


# ----------------------------------------------------------------------------------------
# Stage 2: pronunciation patch (runs every time so the word table is current)
# ----------------------------------------------------------------------------------------
def ensure_pronunciations(word, table):
    import pronouncing

    missing = [w for w in word.lower().split() if not pronouncing.phones_for_word(w) and w not in table]
    if missing:
        sys.exit(f"\n❌ Not in CMUdict and no custom pronunciation: {missing}\n"
                 f"   Re-run with e.g.  --pron \"{missing[0]}=B UW1 M IY0\"  (ARPAbet, stress digits on vowels)\n"
                 f"   or respell with dictionary words, e.g. --word \"boo me\".")

    # Delete the 403 error page that was saved as the DeepPhonemizer checkpoint
    for bad in (OWW_PKG / "resources" / "en_us_cmudict_forward.pt", OWW_PKG / "en_us_cmudict_forward.pt"):
        if bad.exists() and bad.stat().st_size < 10_000:
            bad.unlink()
            print(f"✓ Removed bad phonemizer checkpoint (S3 403 page): {bad}")

    # Patch data.py so custom words bypass DeepPhonemizer (idempotent)
    data_py = OWW_PKG / "data.py"
    src = data_py.read_text()
    if "CUSTOM_PHONES.get" not in src:
        target = "pronouncing.phones_for_word(i)"
        if target not in src:
            sys.exit("❌ data.py layout differs; grep for phones_for_word and patch by hand.")
        src = src.replace(target, "(CUSTOM_PHONES.get(i.lower()) or pronouncing.phones_for_word(i))")
    if not re.search(r"^\s*CUSTOM_PHONES = \{", src, flags=re.M):
        src += "\n\nCUSTOM_PHONES = {}  # oww_train.py custom pronunciations\n"   # module-level, set at import
    src = re.sub(r"CUSTOM_PHONES = \{.*?\}", lambda _: f"CUSTOM_PHONES = {table!r}", src, count=1)
    data_py.write_text(src)
    print(f"✓ data.py custom pronunciations: {table}")


# ----------------------------------------------------------------------------------------
# Stage 3: TTS smoke test
# ----------------------------------------------------------------------------------------
def tts_test(word):
    log(f"TTS test: synthesizing '{word}' -> test_generation.wav")
    sys.path.insert(0, str(PIPER))
    from generate_samples import generate_samples

    generate_samples(text=word, max_samples=1, length_scales=[1.1], noise_scales=[0.7],
                     noise_scale_ws=[0.7], output_dir="./", batch_size=1,
                     auto_reduce_batch_size=True, file_names=["test_generation.wav"])
    require("test_generation.wav")
    print("✓ test_generation.wav written. Preview it in notebook Step 8.")


# ----------------------------------------------------------------------------------------
# Stage 4: background / RIR / precomputed feature data (idempotent)
# ----------------------------------------------------------------------------------------
HF = "https://huggingface.co/datasets"
AUDIOSET_URL = f"{HF}/agkphysics/AudioSet/resolve/main/data/bal_train/09.parquet"            # ~690 MB, FLAC
FMA_URL = f"{HF}/benjamin-paine/free-music-archive-small/resolve/main/data/train-{{:05d}}-of-00015.parquet"  # ~480 MB each, MP3
FEATURES_URL = f"{HF}/davidscripka/openwakeword_features/resolve/main"


def download(url, dest):
    """wget to dest.part (resumable with -c), rename on success. Never leaves a broken dest file."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"✓ {dest.name} already downloaded")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    run(f"wget -c -q --show-progress --progress=dot:giga -O '{part}' '{url}'")
    part.rename(dest)
    return dest


def quiet_logs():
    import logging
    logging.getLogger().setLevel(logging.INFO)
    for name in ("fsspec", "httpcore", "httpx", "urllib3", "huggingface_hub", "filelock", "datasets", "asyncio"):
        logging.getLogger(name).setLevel(logging.WARNING)


def download_data(fma_hours):
    import io
    from math import gcd

    import numpy as np
    import pyarrow.parquet as papq
    import scipy.io.wavfile
    import soundfile as sf
    from scipy.signal import resample_poly
    from tqdm import tqdm

    quiet_logs()

    def to_wav(path, y):
        y = np.clip(np.asarray(y, dtype=np.float32), -1.0, 1.0)
        scipy.io.wavfile.write(path, 16000, (y * 32767).astype(np.int16))

    def decode_16k_mono(b):
        try:
            y, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=True)
            y = y.mean(axis=1)
        except Exception:
            try:
                import librosa
                y, sr = librosa.load(io.BytesIO(b), sr=None, mono=True)
            except Exception:
                return None
        if sr != 16000:
            g = gcd(int(sr), 16000)
            y = resample_poly(y, 16000 // g, int(sr) // g)
        return y

    def parquet_to_wavs(pq, out_dir, prefix, name_col=None, limit=None, start=0):
        """Decode the 'audio' struct column of a Parquet file to 16 kHz mono WAVs. Skips bad rows."""
        pf = papq.ParquetFile(pq)
        cols = ["audio"] + ([name_col] if name_col and name_col in pf.schema_arrow.names else [])
        total = pf.metadata.num_rows if limit is None else min(limit, pf.metadata.num_rows)
        n = skipped = 0
        with tqdm(total=total, desc=Path(pq).name) as bar:
            for batch in pf.iter_batches(batch_size=8, columns=cols):
                d = batch.to_pydict()
                for i, a in enumerate(d["audio"]):
                    if limit is not None and n >= limit:
                        return n
                    y = decode_16k_mono(a["bytes"]) if a and a.get("bytes") else None
                    if y is None or len(y) < 1600:
                        skipped += 1
                        continue
                    name = d[name_col][i] if len(cols) > 1 else f"{prefix}_{start + n:05d}"
                    to_wav(out_dir / f"{name}.wav", y)
                    n += 1
                    bar.update(1)
        if skipped:
            print(f"  (skipped {skipped} undecodable clips)")
        return n

    # --- Room impulse responses (echo) ---
    log("Data: MIT room impulse responses")
    rir_dir = RIR_DIR
    rir_done = done_marker(rir_dir)   # ./.mit_rirs.complete (outside the folder)
    if not rir_done.exists() and rir_dir.exists() and len(list(rir_dir.glob("*.wav"))) >= 250:
        rir_done.touch()   # finished by an earlier run
    if rir_done.exists():
        print(f"✓ mit_rirs ready ({len(list(rir_dir.glob('*.wav')))} files)")
    else:
        import datasets
        rir_dir.mkdir(exist_ok=True)
        ds = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
        for row in tqdm(ds, desc="RIRs"):
            to_wav(rir_dir / row["audio"]["path"].split("/")[-1], row["audio"]["array"])
        rir_done.touch()

    # --- AudioSet (background noise) ---
    log("Data: AudioSet sample (background noise)")
    as16 = AUDIOSET_DIR
    as_done = done_marker(as16)       # ./.audioset_16k.complete (outside the folder)
    if as_done.exists():
        print(f"✓ audioset_16k ready ({len(list(as16.glob('*.wav')))} files)")
    else:
        stale = ROOT / "audioset" / "bal_train09.tar"   # empty file left by the old, broken URL
        if stale.exists() and stale.stat().st_size == 0:
            stale.unlink()
        pq = download(AUDIOSET_URL, ROOT / "audioset" / "bal_train_09.parquet")
        as16.mkdir(exist_ok=True)
        n = parquet_to_wavs(pq, as16, "audioset", name_col="video_id")
        if n == 0:
            sys.exit("❌ No AudioSet clips could be decoded.")
        as_done.touch()
        pq.unlink()   # free ~690 MB
        print(f"✓ audioset_16k: {n} clips")

    # --- Free Music Archive (background music) ---
    need = max(1, int(fma_hours * 3600 // 30))   # FMA-small clips are 30 s
    log(f"Data: Free Music Archive ({fma_hours} h ≈ {need} clips)")
    fma = FMA_DIR
    fma_done = done_marker(fma)       # ./.fma.complete (outside the folder)
    if fma_done.exists():
        print(f"✓ fma ready ({len(list(fma.glob('*.wav')))} files)")
    else:
        fma.mkdir(exist_ok=True)
        have = len(list(fma.glob("*.wav")))
        shard = 0
        while have < need and shard < 15:
            pq = download(FMA_URL.format(shard), ROOT / "fma_raw" / f"train-{shard:05d}.parquet")
            have += parquet_to_wavs(pq, fma, "fma", limit=need - have, start=have)
            pq.unlink()   # free ~480 MB
            shard += 1
        if have == 0:
            sys.exit("❌ No FMA clips could be decoded.")
        fma_done.touch()
        print(f"✓ fma: {have} clips")

    # --- Precomputed negative features + false-positive validation set ---
    log("Data: precomputed openWakeWord features (~16 GB, the big one)")
    for f in ["validation_set_features.npy", "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"]:
        download(f"{FEATURES_URL}/{f}", ROOT / f)


# ----------------------------------------------------------------------------------------
# Stage 5: config + generate / augment / train / convert
# ----------------------------------------------------------------------------------------
def write_config(args):
    import yaml

    config = yaml.load(open(OWW / "examples" / "custom_model.yml").read(), yaml.Loader)
    config["target_phrase"] = [args.word]
    config["model_name"] = args.word.replace(" ", "_")
    config["n_samples"] = args.examples
    config["n_samples_val"] = max(500, args.examples // 10)
    config["steps"] = args.steps
    config["target_accuracy"] = 0.5
    config["target_recall"] = 0.25
    config["output_dir"] = str(OUT)
    config["max_negative_weight"] = args.penalty
    config["rir_paths"] = ["./mit_rirs"]
    config["background_paths"] = ["./audioset_16k", "./fma"]
    config["false_positive_validation_data_path"] = "validation_set_features.npy"
    config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
    with open("my_model.yaml", "w") as f:
        yaml.dump(config, f)
    return config["model_name"]


def train(args, mn):
    train_py = f"{sys.executable} {OWW_PKG / 'train.py'} --training_config my_model.yaml"
    model_dir = OUT / mn
    feats = [model_dir / f"{k}_features_{s}.npy" for k in ("positive", "negative") for s in ("train", "test")]

    log("Train 1/4: generate clips (existing clips are reused)")
    run(f"{train_py} --generate_clips")

    log("Train 2/4: augment clips + compute features")
    if args.reuse_features and all(p.exists() for p in feats):
        print("✓ Reusing existing feature files")
    else:
        sanitize_audio_dirs(AUDIO_DIRS)   # train.py loads every file in these folders as audio
        for p in model_dir.glob("*.npy"):
            p.unlink()   # stale partial features cause a silent skip, then a crash in training
        run(f"{train_py} --augment_clips")
    require(*feats)

    log("Train 3/4: train model")
    run(f"{train_py} --train_model")
    onnx = OUT / f"{mn}.onnx"
    require(onnx)

    log("Train 4/4: convert ONNX -> TFLite")
    tflite = OUT / f"{mn}.tflite"
    if args.skip_tflite:
        print("Skipping TFLite conversion (--skip-tflite)")
    elif run(f"onnx2tf -i {onnx} -o {OUT}/ -kat onnx____Flatten_0", check=False) \
            and (OUT / f"{mn}_float32.tflite").exists():
        shutil.move(OUT / f"{mn}_float32.tflite", tflite)
    else:
        print("⚠️  TFLite conversion failed; the ONNX model is still usable.")

    log("✅ Done")
    print(f"ONNX:   {onnx}")
    if tflite.exists():
        print(f"TFLite: {tflite}")


# ----------------------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------------------
def parse_args():
    p = argparse.ArgumentParser(description="Train a custom openWakeWord model end to end.")
    p.add_argument("--word", help='Target wake word/phrase, e.g. "hey spark" (required unless --data-only)')
    p.add_argument("--pron", action="append", default=[],
                   help='Custom ARPAbet for an out-of-dictionary word: "kyla=K AY1 L AH0" (repeatable)')
    p.add_argument("--examples", type=int, default=1000, help="Synthetic positive examples (100-50000)")
    p.add_argument("--steps", type=int, default=10000, help="Training steps (0-50000)")
    p.add_argument("--penalty", type=int, default=1500, help="False-activation penalty (100-5000)")
    p.add_argument("--fma-hours", type=float, default=1.0, help="Hours of FMA music to download")
    p.add_argument("--test-only", action="store_true", help="Only synthesize test_generation.wav")
    p.add_argument("--data-only", action="store_true", help="Only download the background/RIR/feature data")
    p.add_argument("--reuse-features", action="store_true", help="Skip augmentation if all feature files exist")
    p.add_argument("--skip-tflite", action="store_true", help="Skip ONNX -> TFLite conversion")
    args = p.parse_args()
    if not args.word and not args.data_only:
        p.error("--word is required (unless --data-only)")
    return args


def main():
    args = parse_args()
    check_setup()
    if args.data_only:
        download_data(args.fma_hours)
        return

    table = dict(CUSTOM_WORDS)
    for spec in args.pron:
        if "=" not in spec:
            sys.exit(f'❌ --pron must look like "word=ARPABET", got: {spec}')
        w, phones = spec.split("=", 1)
        table[w.strip().lower()] = [phones.strip()]
    ensure_pronunciations(args.word, table)

    import torch
    print(f"Device: {'GPU ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slow! switch runtime to GPU)'}")

    tts_test(args.word)
    if args.test_only:
        return

    download_data(args.fma_hours)   # no-op if Step 9 already ran
    mn = write_config(args)
    train(args, mn)


if __name__ == "__main__":
    main()

In [ ]:
# @title 6 · Settings  (change the wake word here, then run this cell)
import shlex

target_word = "hey spark"  # @param {type:"string"}
# @markdown Spell it how it sounds. Use spaces between words; no punctuation except `?` and `!`.

custom_pronunciation = ""  # @param {type:"string"}
# @markdown Only needed if Step 7 fails with **"Not in CMUdict"**. Format: `word=ARPABET`, e.g. `kyla=K AY1 L AH0`.

number_of_examples = 5100  # @param {type:"slider", min:100, max:50000, step:50}
number_of_training_steps = 11000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 1500  # @param {type:"slider", min:100, max:5000, step:50}
music_hours = 3  # @param {type:"slider", min:1, max:10, step:1}

target_word = target_word.strip()
assert target_word, "Enter a wake word."
MODEL_NAME = target_word.replace(" ", "_")

BASE_ARGS = f"--word {shlex.quote(target_word)}"
if custom_pronunciation.strip():
    assert "=" in custom_pronunciation, "custom_pronunciation must look like  word=ARPABET"
    BASE_ARGS += f" --pron {shlex.quote(custom_pronunciation.strip())}"
DATA_ARGS = f"--data-only --fma-hours {music_hours}"
TRAIN_ARGS = (f"{BASE_ARGS} --examples {number_of_examples} --steps {number_of_training_steps} "
              f"--penalty {false_activation_penalty} --fma-hours {music_hours}")

print("Wake word :", target_word)
print("Model name:", MODEL_NAME)
print("Command   : python oww_train.py", TRAIN_ARGS)

In [ ]:
# @title 7 · Synthesize a test clip  (~1 min)
# @markdown Makes one sample of your wake word with the training voice and saves it as
# @markdown `test_generation.wav`. Listen to it in Step 8.
!python oww_train.py {BASE_ARGS} --test-only

In [ ]:
# @title 8 · Preview the voice
# @markdown **Listen to it.** If it doesn't sound like your wake word, change the spelling
# @markdown (or add a `custom_pronunciation`) in Step 6 and run Steps 6–8 again.
import os
from IPython.display import Audio, display

if os.path.exists("test_generation.wav"):
    display(Audio("test_generation.wav", autoplay=True))
else:
    print("❌ No test clip yet. Run Step 7, and if it fails scroll up to the line starting with ❌.")

In [ ]:
# @title 9 · Download the training data  (~20–40 min, first time only)
# @markdown Room echoes (MIT RIRs), background noise (AudioSet), music (Free Music Archive,
# @markdown `music_hours` from Step 6), and openWakeWord's precomputed negative features
# @markdown (**~16 GB**, the big one). Re-running skips anything already downloaded.
!python oww_train.py {DATA_ARGS}

In [ ]:
# @title 10 · Train the model  (~30–60 min on a T4)
# @markdown Generates clips, augments them, trains, and converts to TFLite. (If you skipped
# @markdown Step 9, it downloads the training data first.)
# @markdown Keep this tab open. If it stops with a **❌** line, copy that line and the error above it.
!python oww_train.py {TRAIN_ARGS}

In [ ]:
# @title 11 · Download the model files
import os
from google.colab import files

found = False
for ext in ("onnx", "tflite"):
    p = f"my_custom_model/{MODEL_NAME}.{ext}"
    if os.path.exists(p):
        print("Downloading", p); files.download(p); found = True
    else:
        print("Not found:", p)
if not found:
    print("❌ No model files. Step 10 didn't finish; scroll up in its output.")

In [ ]:
# @title 12 · Bundle the ONNX model into one file  (only if needed)
# @markdown Newer PyTorch can save the weights next to the model as `<name>.onnx.data`, so the
# @markdown `.onnx` from Step 11 won't load on its own. If that happened, this writes
# @markdown `<name>_single.onnx` with the weights embedded: that's the one file to copy into
# @markdown Hey Spark's `assets/wakewords/`. Otherwise it does nothing.
import onnx
from pathlib import Path
from google.colab import files

src = Path(f"my_custom_model/{MODEL_NAME}.onnx")
assert src.exists(), f"{src} not found. Did Step 10 finish?"

if not src.with_name(src.name + ".data").exists():
    print(f"✅ {src.name} is already a single file. Nothing to do.")
else:
    model = onnx.load(str(src))                 # also pulls in <name>.onnx.data
    onnx.checker.check_model(model)
    single = src.with_name(f"{MODEL_NAME}_single.onnx")
    onnx.save(model, str(single), save_as_external_data=False)   # one self-contained file
    print(f"saved {single}: {single.stat().st_size // 1024} KB")
    files.download(str(single))


In [ ]:
# @title 13 · (Optional) Save a copy to Google Drive
# @markdown Colab wipes its files when the session ends. Tick this box and run the cell to keep
# @markdown the models in **My Drive → wakeword_models**. It asks for permission to access your
# @markdown Drive.
save_to_drive = False  # @param {type:"boolean"}

import os, shutil

if not save_to_drive:
    print("Skipped. Tick save_to_drive and run this cell again to save to Google Drive.")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/wakeword_models"
    os.makedirs(dest, exist_ok=True)
    for name in (f"{MODEL_NAME}.onnx", f"{MODEL_NAME}_single.onnx", f"{MODEL_NAME}.tflite"):
        p = f"my_custom_model/{name}"
        if os.path.exists(p):
            shutil.copy(p, dest); print("Saved", os.path.join(dest, os.path.basename(p)))
